# LicitaIA — Entrenamiento del Modelo Random Forest
**MS2 — ML Service**

Este notebook entrena el modelo de predicción con los datos reales del SECOP II.

**Requisitos previos:**
1. `python data/secop_loader.py` — descarga los datos
2. `python data/cleaner.py` — limpia el dataset
3. Ejecuta este notebook celda por celda

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder

print(' Librerías importadas')

In [ ]:
# Cargar dataset limpio
df = pd.read_csv('data/processed/secop_limpio.csv', low_memory=False)
print(f'Dataset cargado: {len(df):,} filas')
print(f'Columnas: {list(df.columns)}')
print(f"\nDistribución objetivo:")
print(df['gano'].value_counts())

In [ ]:
# Preparar features — MISMO ORDEN que en feature_engineer.py
SECTORES_COMPETIDOS = ['OBRA', 'CONSULTORÍA', 'SUMINISTRO', 'INTERVENTORÍA', 'PRESTACIÓN DE SERVICIOS']
MODALIDADES_ABIERTAS = ['LICITACIÓN PÚBLICA', 'CONCURSO DE MÉRITOS', 'SELECCIÓN ABREVIADA']
ENTIDADES_ALTO_VOLUMEN = ['MINISTERIO', 'GOBERNACIÓN', 'ALCALDÍA', 'HOSPITAL', 'ESE', 'ICBF']

df['sector_u']    = df['tipo_de_contrato'].fillna('').str.upper()
df['modalidad_u'] = df['modalidad_de_contratacion'].fillna('').str.upper()
df['entidad_u']   = df['nombre_entidad'].fillna('').str.upper()

# Construir la matriz X de features
X = pd.DataFrame({
    'log_cuantia':          np.log1p(df['cuantia_contrato']),
    'cuantia_rango_medio':  ((df['cuantia_contrato'] >= 50e6) & (df['cuantia_contrato'] <= 800e6)).astype(int),
    'cuantia_muy_alta':     (df['cuantia_contrato'] > 2000e6).astype(int),
    'sector_competido':     df['sector_u'].apply(lambda s: int(any(x in s for x in SECTORES_COMPETIDOS))),
    'modalidad_abierta':    df['modalidad_u'].apply(lambda m: int(any(x in m for x in MODALIDADES_ABIERTAS))),
    'entidad_alto_volumen': df['entidad_u'].apply(lambda e: int(any(x in e for x in ENTIDADES_ALTO_VOLUMEN))),
    'tiene_nit':            df['nit_entidad'].notna().astype(int),
    'tiene_municipio':      df['ciudad_entidad'].notna().astype(int),
})

y = df['gano']

print(f'Features construidos: {X.shape}')
print(X.head())

In [ ]:
# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Entrenamiento: {len(X_train):,} filas')
print(f'Prueba:        {len(X_test):,} filas')

In [ ]:
# Entrenar Random Forest
modelo = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=20,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
)
modelo.fit(X_train, y_train)
print(' Modelo entrenado')

In [ ]:
# Evaluar el modelo
y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]

print('MÉTRICAS DEL MODELO')
print('─' * 40)
print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'AUC-ROC:   {roc_auc_score(y_test, y_prob):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No ganó', 'Ganó']))

# Importancia de features
importancias = pd.Series(modelo.feature_importances_, index=X.columns)
print('\nIMPORTANCIA DE FEATURES:')
print(importancias.sort_values(ascending=False))

In [ ]:
# Guardar el modelo
Path('models').mkdir(exist_ok=True)
joblib.dump(modelo, 'models/random_forest.pkl')
print(' Modelo guardado en models/random_forest.pkl')
print('\n El MS2 cargará este modelo automáticamente al reiniciarse')
print(' Reinicia el MS2 con: uvicorn app.main:app --reload --port 8000')